# NB2 — Spatial Denoising of Raw Transmission Cube

## NB1 Recap

NB1 established **T-domain raw fitting** as the best per-pixel method for composition maps:
- MAE at n=2: **140,976 ppm**; at n=10: **81,839 ppm**
- TV regularization on composition maps reduced MAE by ~24% but introduced edge blurring
- **Critical finding**: Preprocessing HURT SAMMY fitting — parametric T-domain reconstruction
  erased resonance fine structure that SAMMY needs for isotope discrimination
  (noisy raw delta=1,956 ppm vs T-domain delta=298,287 ppm at Pure U-235 n=10)

## NB2 Hypothesis

**Spatially denoising the raw transmission cube *before* fitting** preserves spectral shape
(good for SAMMY) while reducing spatial noise (good for composition maps). This is
fundamentally different from NB1's approach of post-processing composition maps.

Each energy bin is denoised as a 2D spatial image — no cross-energy smoothing, so
resonance peaks are preserved.

## Experiments

| Exp | Method | Package | Key Parameter |
|-----|--------|---------|---------------|
| 2a | Gaussian smoothing | `scipy.ndimage.gaussian_filter` | sigma = [0.5, 1, 2, 3, 5] px |
| 2b | Non-Local Means | `skimage.restoration.denoise_nl_means` | h = [0.05, 0.1, 0.2, 0.3, 0.5] |
| 2c | Wavelet BayesShrink | `skimage.restoration.denoise_wavelet` | wavelet = [db1, db2, sym4] |

### SAMMY Validation States (per pixel, per noise level)
1. Clean (reference)
2. Noisy raw (baseline)
3. Spatially-denoised raw → SAMMY
4. Spatially-denoised → T-domain recon → SAMMY

In [ ]:
import os
os.environ["TQDM_DISABLE"] = "1"

import time
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.optimize import least_squares
from scipy.optimize import nnls as scipy_nnls
from scipy.ndimage import gaussian_filter
from skimage.restoration import denoise_nl_means, denoise_wavelet, estimate_sigma

from pleiades.imaging import (
    HyperspectralLoader,
    DataDegrader,
    PhysicsRecovery,
    HyperspectralData,
)
from pleiades.imaging.config import ImagingConfig
from pleiades.utils.logger import configure_logger

configure_logger(console_level="WARNING")

DEBUG_DIR = Path("../../_debug_images")
DEBUG_DIR.mkdir(exist_ok=True)


def save_fig(fig, name):
    fig.savefig(DEBUG_DIR / f"{name}.png", dpi=150, bbox_inches="tight")
    plt.show()


print("Imports OK")

In [ ]:
# --- Representative pixels (identified from clean NNLS ground truth) ---
REP_PIXELS = {
    "Pure U-235": (155, 47),
    "Pure Pu-241": (112, 174),
    "Overlap": (136, 223),
}


PPM = 1_000_000  # Conversion factor: fraction -> ppm


def plot_abundance_maps(abundance_maps, isotope_names, title, success_mask=None,
                        vmin=0, vmax=PPM, units_label="Composition (ppm)"):
    """Plot composition maps in ppm for each isotope."""
    n_iso = abundance_maps.shape[0]
    fig, axes = plt.subplots(1, n_iso, figsize=(6 * n_iso, 5))
    if n_iso == 1:
        axes = [axes]
    for i, iso in enumerate(isotope_names):
        amap = abundance_maps[i].copy() * PPM
        if success_mask is not None:
            amap[~success_mask] = np.nan
        im = axes[i].imshow(amap, cmap="viridis", origin="upper",
                            vmin=vmin, vmax=vmax)
        axes[i].set_title(iso)
        fig.colorbar(im, ax=axes[i], shrink=0.8, label=units_label)
    fig.suptitle(title, y=1.02)
    fig.tight_layout()
    return fig


def plot_difference_maps(abundance_maps, gt_maps, isotope_names, title,
                         success_mask=None):
    """Plot difference (method - ground truth) in ppm for each isotope."""
    n_iso = abundance_maps.shape[0]
    fig, axes = plt.subplots(1, n_iso, figsize=(6 * n_iso, 5))
    if n_iso == 1:
        axes = [axes]
    for i, iso in enumerate(isotope_names):
        diff = (abundance_maps[i] - gt_maps[i]) * PPM
        if success_mask is not None:
            diff[~success_mask] = np.nan
        vmax_ppm = max(300_000, np.nanmax(np.abs(diff[np.isfinite(diff)])))
        im = axes[i].imshow(diff, cmap="RdBu_r", origin="upper",
                            vmin=-vmax_ppm, vmax=vmax_ppm)
        axes[i].set_title(f"{iso} difference (ppm)")
        fig.colorbar(im, ax=axes[i], shrink=0.8, label="Composition error (ppm)")
    fig.suptitle(f"{title} (red=overestimate, blue=underestimate)", y=1.02)
    fig.tight_layout()
    return fig


def plot_spectra_at_pixels(energy, data_cubes, labels, colors=None,
                           title_suffix=""):
    """Plot transmission spectra at 3 representative pixels.

    data_cubes: list of (n_energy, H, W) arrays
    labels: list of string labels
    colors: optional list of colors
    """
    if colors is None:
        default_colors = ["black", "#d62728", "#2ca02c", "#1f77b4",
                          "#ff7f0e", "#9467bd", "#e377c2"]
        colors = [default_colors[i % len(default_colors)]
                  for i in range(len(labels))]

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    for col, (pix_label, (pr, pc)) in enumerate(REP_PIXELS.items()):
        ax_top = axes[0, col]
        for idx, (cube, lbl) in enumerate(zip(data_cubes, labels)):
            y = cube[:, pr, pc]
            if idx == 0:
                ax_top.plot(energy, y, lw=1.2, color=colors[idx],
                           label=lbl, zorder=10)
            else:
                ax_top.scatter(energy, y, s=1, alpha=0.5,
                              color=colors[idx], label=lbl, zorder=5 - idx)
        ax_top.set_title(f"{pix_label} ({pr},{pc})")
        ax_top.set_ylabel("Transmission")
        ax_top.set_ylim(-0.05, 1.15)
        ax_top.legend(fontsize=7, markerscale=5)

        ax_bot = axes[1, col]
        clean = data_cubes[0][:, pr, pc]
        for idx, (cube, lbl) in enumerate(zip(data_cubes[1:], labels[1:]), 1):
            residual = cube[:, pr, pc] - clean
            ax_bot.scatter(energy, residual, s=1, alpha=0.5,
                          color=colors[idx], label=lbl)
        ax_bot.axhline(0, color="black", lw=0.5, ls="--")
        ax_bot.set_title("Residual vs clean")
        ax_bot.set_ylabel("T(method) - T(clean)")
        ax_bot.set_xlabel("Energy (eV)")
        ax_bot.legend(fontsize=7, markerscale=5)

    fig.suptitle(f"Spectral Comparison{title_suffix}", y=1.02)
    fig.tight_layout()
    return fig


print(f"Helpers loaded. Representative pixels: {REP_PIXELS}")

In [ ]:
# --- T-Domain and NNLS fitting functions ---

def compute_poisson_sigma(T_obs, n_incident):
    """Poisson noise std-dev in transmission domain.

    sigma = sqrt(max(T, 1/n) / n), floored at 1/n (one-count uncertainty).
    """
    T_safe = np.maximum(T_obs, 1.0 / n_incident)
    sigma = np.sqrt(T_safe / n_incident)
    return np.maximum(sigma, 1.0 / n_incident)


def fit_pixel_tdomain(T_obs, sigma, A_matrix, x0=None):
    """Fit one pixel in transmission domain via nonlinear least squares.

    Minimizes: sum_e [(T_obs(e) - exp(-A @ c)) / sigma(e)]^2
    subject to: c >= 0

    Uses scipy.optimize.least_squares with trust-region reflective
    algorithm and analytical Jacobian.

    Returns: (coefficients, cost, success)
    """
    n_iso = A_matrix.shape[1]
    if x0 is None:
        x0 = np.full(n_iso, 0.1)

    sigma_safe = np.maximum(sigma, 1e-10)

    def residual(c):
        T_model = np.exp(-A_matrix @ c)
        return (T_obs - T_model) / sigma_safe

    def jacobian(c):
        T_model = np.exp(-A_matrix @ c)
        # d(residual_e)/d(c_i) = A[e,i] * exp(-A @ c) / sigma[e]
        return (A_matrix * T_model[:, np.newaxis]) / sigma_safe[:, np.newaxis]

    try:
        result = least_squares(
            residual, x0, jac=jacobian, method="trf",
            bounds=(0, np.inf), max_nfev=200,
        )
        return result.x, result.cost, result.success
    except Exception:
        return np.zeros(n_iso), np.nan, False


def fit_image_tdomain(data_cube, sigma_cube, A_matrix, x0_map=None, label=""):
    """Fit all pixels in transmission domain.

    Parameters
    ----------
    data_cube : (n_energy, H, W)
    sigma_cube : (n_energy, H, W)
    A_matrix : (n_energy, n_isotopes)
    x0_map : optional (n_isotopes, H, W) initial guesses
    label : progress label

    Returns
    -------
    coeff_maps : (n_isotopes, H, W)
    cost_map : (H, W)
    success_mask : (H, W) bool
    """
    n_e, H, W = data_cube.shape
    n_iso = A_matrix.shape[1]
    coeff_maps = np.full((n_iso, H, W), np.nan)
    cost_map = np.full((H, W), np.nan)
    success_mask = np.zeros((H, W), dtype=bool)

    t0 = time.time()
    for r in range(H):
        for c in range(W):
            T_obs = data_cube[:, r, c]
            sig = sigma_cube[:, r, c]
            x0 = x0_map[:, r, c] if x0_map is not None else None
            coeffs, cost, ok = fit_pixel_tdomain(T_obs, sig, A_matrix, x0)
            if ok:
                coeff_maps[:, r, c] = coeffs
                cost_map[r, c] = cost
                success_mask[r, c] = True
        if (r + 1) % 64 == 0:
            elapsed = time.time() - t0
            rate = (r + 1) * W / elapsed
            eta = (H - r - 1) * W / rate
            print(f"  {label} row {r+1}/{H} "
                  f"({rate:.0f} px/s, ETA {eta:.0f}s)")

    elapsed = time.time() - t0
    n_ok = int(success_mask.sum())
    print(f"  {label} done: {n_ok}/{H*W} pixels in {elapsed:.1f}s")
    return coeff_maps, cost_map, success_mask


def coeffs_to_abundances(coeff_maps, success_mask):
    """Normalize coefficients to composition fractions (sum to 1 per pixel)."""
    abundance = coeff_maps.copy()
    total = np.nansum(abundance, axis=0)
    nonzero = (total > 0) & success_mask
    for i in range(abundance.shape[0]):
        abundance[i, nonzero] /= total[nonzero]
    zero = (total == 0) | ~success_mask
    for i in range(abundance.shape[0]):
        abundance[i, zero] = np.nan
    return abundance


def fit_pixel_nnls(T_obs, sigma, A_matrix):
    """Baseline NNLS in attenuation domain (for comparison).

    Converts T_obs -> A_obs = -ln(clip(T, 1e-6, 1-1e-6)),
    then solves weighted NNLS: A_obs = A_matrix @ c, c >= 0.

    Returns: (coefficients, success)
    """
    T_clamp = np.clip(T_obs, 1e-6, 1.0 - 1e-6)
    A_obs = -np.log(T_clamp)
    sigma_abs = np.maximum(sigma, 1e-10) / T_clamp
    weights = 1.0 / np.maximum(sigma_abs, 1e-10)
    try:
        c, _ = scipy_nnls(weights[:, np.newaxis] * A_matrix, weights * A_obs)
        return c, True
    except Exception:
        return np.zeros(A_matrix.shape[1]), False


def fit_image_nnls(data_cube, sigma_cube, A_matrix, label=""):
    """NNLS in attenuation domain for all pixels."""
    n_e, H, W = data_cube.shape
    n_iso = A_matrix.shape[1]
    coeff_maps = np.full((n_iso, H, W), np.nan)
    success_mask = np.zeros((H, W), dtype=bool)
    t0 = time.time()
    for r in range(H):
        for c in range(W):
            coeffs, ok = fit_pixel_nnls(
                data_cube[:, r, c], sigma_cube[:, r, c], A_matrix)
            if ok:
                coeff_maps[:, r, c] = coeffs
                success_mask[r, c] = True
    elapsed = time.time() - t0
    n_ok = int(success_mask.sum())
    print(f"  {label} NNLS done: {n_ok}/{H*W} in {elapsed:.1f}s")
    return coeff_maps, success_mask


def compute_mae(pred, gt, mask=None):
    """Mean absolute error over valid pixels."""
    if mask is None:
        mask = np.isfinite(pred) & np.isfinite(gt)
    else:
        mask = mask & np.isfinite(pred) & np.isfinite(gt)
    if mask.sum() == 0:
        return np.nan
    return float(np.mean(np.abs(pred[mask] - gt[mask])))


def compute_rmse(pred, gt, mask=None):
    """Root mean square error over valid pixels."""
    if mask is None:
        mask = np.isfinite(pred) & np.isfinite(gt)
    else:
        mask = mask & np.isfinite(pred) & np.isfinite(gt)
    if mask.sum() == 0:
        return np.nan
    return float(np.sqrt(np.mean((pred[mask] - gt[mask]) ** 2)))


print("T-domain fitting functions defined.")

In [ ]:
# --- Spatial denoising functions for transmission cubes ---
# Each function denoises per energy bin as a 2D spatial image.
# No cross-energy smoothing — resonance peaks preserved.

def denoise_cube_gaussian(cube, sigma_spatial):
    """Gaussian spatial smoothing of each energy slice.

    Parameters
    ----------
    cube : (n_energy, H, W) transmission cube
    sigma_spatial : float, Gaussian kernel width in pixels

    Returns
    -------
    denoised : (n_energy, H, W)
    """
    denoised = np.empty_like(cube)
    for e in range(cube.shape[0]):
        denoised[e] = gaussian_filter(cube[e], sigma=sigma_spatial)
    return denoised


def denoise_cube_nlmeans(cube, h, patch_size=5, patch_distance=4, fast_mode=True):
    """Non-local means denoising of each energy slice.

    Parameters
    ----------
    cube : (n_energy, H, W) transmission cube
    h : float, filtering strength (higher = more smoothing)
    patch_size : int, patch size for comparison (odd number)
    patch_distance : int, search window radius
    fast_mode : bool, use fast approximation

    Returns
    -------
    denoised : (n_energy, H, W)
    """
    denoised = np.empty_like(cube)
    for e in range(cube.shape[0]):
        sigma_est = estimate_sigma(cube[e])
        denoised[e] = denoise_nl_means(
            cube[e], h=h * sigma_est, sigma=sigma_est,
            patch_size=patch_size, patch_distance=patch_distance,
            fast_mode=fast_mode,
        )
    return denoised


def denoise_cube_wavelet(cube, wavelet="db1"):
    """Wavelet BayesShrink denoising of each energy slice.

    Parameters
    ----------
    cube : (n_energy, H, W) transmission cube
    wavelet : str, wavelet family name (e.g. 'db1', 'db2', 'sym4')

    Returns
    -------
    denoised : (n_energy, H, W)
    """
    denoised = np.empty_like(cube)
    for e in range(cube.shape[0]):
        denoised[e] = denoise_wavelet(
            cube[e], wavelet=wavelet, method="BayesShrink",
            mode="soft", rescale_sigma=True,
        )
    return denoised


print("Spatial denoising functions defined: Gaussian, NLM, Wavelet")

In [ ]:
# --- Config, data loading, noise generation, baselines ---

sammy_executable = Path("/Users/8cz/code-int.ornl.gov/sammy/build/bin/sammy")
tiff_path = Path("../../tests/data/pleiades_data/LANL-ORNL_example.tif")
energy = np.linspace(1.0, 50.0, 500)
isotope_names = ["U-235", "Pu-241"]

config = ImagingConfig(
    isotopes=isotope_names,
    element="U",
    mass_number=235,
    density_g_cm3=19.1,
    thickness_mm=0.21,
    atomic_mass_amu=235.0439,
    natural_abundances=False,
    custom_abundances=[0.5, 0.5],
    min_energy_eV=1.0,
    max_energy_eV=50.0,
    temperature_K=293.6,
    fit_abundances=True,
)

# --- Load clean data ---
loader = HyperspectralLoader(tiff_path, energy=energy)
hyperspectral = loader.load()

hyperspectral.uncertainty = np.maximum(
    hyperspectral.uncertainty,
    0.02 * np.abs(hyperspectral.data) + 1e-4,
).astype(np.float32)

n_energy, height, width = hyperspectral.shape
print(f"Shape: {height}x{width} pixels, {n_energy} energy bins")

# --- Generate SAMMY reference spectra ---
recovery = PhysicsRecovery(imaging_config=config, sammy_executable=sammy_executable)
ref_spectra = recovery.generate_reference_spectra(energy)
print(f"Reference spectra: {[r.isotope_name for r in ref_spectra]}")
for ref in ref_spectra:
    print(f"  {ref.isotope_name}: T range "
          f"[{ref.transmission.min():.4f}, {ref.transmission.max():.4f}]")

# --- Build absorption matrix A (n_energy, n_isotopes) ---
A_matrix = np.column_stack([r.absorption for r in ref_spectra])
print(f"\nA_matrix shape: {A_matrix.shape}")
print(f"A_matrix range: [{A_matrix.min():.4f}, {A_matrix.max():.4f}]")

# --- Generate noisy datasets ---
noise_levels = [2, 10]  # Focus on n=2 (extreme) and n=10 (moderate)
noisy_data = {}
sigma_data = {}
sigma_clean = 0.01 * np.ones_like(hyperspectral.data)

print(f"\n{'n_incident':>10} {'Zero-T pixels':>15} {'Max T_obs':>10}")
print("-" * 40)
for n_inc in noise_levels:
    degrader = DataDegrader(random_seed=42)
    data_noisy = degrader.add_poisson_noise(hyperspectral.data, n_incident=n_inc)
    noisy_data[n_inc] = data_noisy
    sigma_data[n_inc] = compute_poisson_sigma(data_noisy, n_inc)
    n_zero = (data_noisy == 0).sum()
    pct_zero = 100.0 * n_zero / data_noisy.size
    print(f"{n_inc:>10} {pct_zero:>14.1f}% {data_noisy.max():>10.3f}")

# --- NNLS reference on clean data (ground truth) ---
print("\nFitting clean NNLS reference...")
gt_coeffs, gt_mask = fit_image_nnls(
    hyperspectral.data, sigma_clean, A_matrix, label="GT")
gt_abundances = coeffs_to_abundances(gt_coeffs, gt_mask)

fig = plot_abundance_maps(gt_abundances, isotope_names,
                          "Ground Truth (Clean NNLS)", gt_mask)
save_fig(fig, "NB2_00_ground_truth")

# --- T-domain raw baselines at n=2 and n=10 ---
raw_results = {}  # n_inc -> (coeff_maps, cost_map, success_mask, abundances)
for n_inc in noise_levels:
    print(f"\nFitting T-domain raw at n={n_inc}...")
    # Use original Poisson sigma (not denoised data sigma)
    coeffs, cost, mask = fit_image_tdomain(
        noisy_data[n_inc], sigma_data[n_inc], A_matrix,
        label=f"Raw n={n_inc}")
    abund = coeffs_to_abundances(coeffs, mask)
    raw_results[n_inc] = (coeffs, cost, mask, abund)
    mae = compute_mae(abund[0], gt_abundances[0], mask & gt_mask) * PPM
    print(f"  Raw T-domain MAE (U-235): {mae:,.0f} ppm")

print("\nSetup complete.")

## Experiment 2a: Gaussian Spatial Smoothing

Gaussian smoothing is the simplest spatial denoising method. Each energy slice is
convolved with a 2D Gaussian kernel of width `sigma` pixels. This blurs the image
uniformly, reducing noise but also reducing spatial resolution.

We sweep `sigma` = [0.5, 1, 2, 3, 5] pixels and pick the best by MAE.

In [ ]:
# --- Exp 2a: Gaussian sigma sweep at n=2 ---
gauss_sigmas = [0.5, 1.0, 2.0, 3.0, 5.0]
gauss_sweep = {}  # sigma -> {n_inc: (coeffs, cost, mask, abund, mae)}

n_inc_sweep = 2  # Sweep at extreme noise
print(f"Gaussian sigma sweep at n={n_inc_sweep}")
print(f"{'sigma':>6} {'MAE U-235 (ppm)':>18} {'MAE Pu-241 (ppm)':>18} {'Time (s)':>10}")
print("-" * 56)

best_sigma = None
best_mae = np.inf

for sig_sp in gauss_sigmas:
    t0 = time.time()
    cube_dn = denoise_cube_gaussian(noisy_data[n_inc_sweep], sig_sp)
    # Use ORIGINAL Poisson sigma for fitting (prevents overfitting to smoothed data)
    coeffs, cost, mask = fit_image_tdomain(
        cube_dn, sigma_data[n_inc_sweep], A_matrix,
        label=f"Gauss s={sig_sp}")
    abund = coeffs_to_abundances(coeffs, mask)
    mae_u = compute_mae(abund[0], gt_abundances[0], mask & gt_mask) * PPM
    mae_p = compute_mae(abund[1], gt_abundances[1], mask & gt_mask) * PPM
    elapsed = time.time() - t0
    print(f"{sig_sp:>6.1f} {mae_u:>18,.0f} {mae_p:>18,.0f} {elapsed:>10.1f}")
    gauss_sweep[sig_sp] = {n_inc_sweep: (coeffs, cost, mask, abund, mae_u)}

    if mae_u < best_mae:
        best_mae = mae_u
        best_sigma = sig_sp

print(f"\nBest sigma: {best_sigma} (MAE={best_mae:,.0f} ppm)")

# --- Sweep plot ---
raw_mae_2 = compute_mae(raw_results[n_inc_sweep][3][0], gt_abundances[0],
                        raw_results[n_inc_sweep][2] & gt_mask) * PPM

fig, ax = plt.subplots(figsize=(8, 5))
maes_u = [gauss_sweep[s][n_inc_sweep][4] for s in gauss_sigmas]
ax.plot(gauss_sigmas, maes_u, "o-", color="#1f77b4", lw=2, markersize=8)
ax.axhline(raw_mae_2, color="red", ls="--", alpha=0.7,
           label=f"Raw baseline ({raw_mae_2:,.0f} ppm)")
ax.set_xlabel("Gaussian sigma (pixels)")
ax.set_ylabel("MAE U-235 (ppm)")
ax.set_title(f"Exp 2a: Gaussian sweep at n={n_inc_sweep}")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
save_fig(fig, "NB2_02a_gauss_sweep")

# --- Spectral overlay at best sigma ---
cube_best_gauss = denoise_cube_gaussian(noisy_data[n_inc_sweep], best_sigma)
fig = plot_spectra_at_pixels(
    energy,
    [hyperspectral.data, noisy_data[n_inc_sweep], cube_best_gauss],
    ["Clean", f"Noisy n={n_inc_sweep}", f"Gauss σ={best_sigma}"],
    title_suffix=f" — Gaussian σ={best_sigma}, n={n_inc_sweep}")
save_fig(fig, "NB2_02a_gauss_spectra")

In [ ]:
# --- Exp 2a: Full-image maps at n=2 and n=10 with best sigma ---
gauss_full = {}  # n_inc -> (coeffs, cost, mask, abund)

for n_inc in noise_levels:
    if n_inc == n_inc_sweep and best_sigma in gauss_sweep:
        # Reuse sweep result
        c, co, m, a, _ = gauss_sweep[best_sigma][n_inc]
        gauss_full[n_inc] = (c, co, m, a)
    else:
        print(f"Fitting Gauss σ={best_sigma} at n={n_inc}...")
        cube_dn = denoise_cube_gaussian(noisy_data[n_inc], best_sigma)
        coeffs, cost, mask = fit_image_tdomain(
            cube_dn, sigma_data[n_inc], A_matrix,
            label=f"Gauss n={n_inc}")
        abund = coeffs_to_abundances(coeffs, mask)
        gauss_full[n_inc] = (coeffs, cost, mask, abund)

for n_inc in noise_levels:
    _, _, mask, abund = gauss_full[n_inc]
    combined_mask = mask & gt_mask

    mae_u = compute_mae(abund[0], gt_abundances[0], combined_mask) * PPM
    mae_p = compute_mae(abund[1], gt_abundances[1], combined_mask) * PPM
    raw_mae_u = compute_mae(raw_results[n_inc][3][0], gt_abundances[0],
                            raw_results[n_inc][2] & gt_mask) * PPM
    pct_improve = (raw_mae_u - mae_u) / raw_mae_u * 100
    print(f"n={n_inc}: Gauss MAE U-235={mae_u:,.0f} ppm "
          f"(raw={raw_mae_u:,.0f}, improvement={pct_improve:+.1f}%)")

    fig = plot_abundance_maps(abund, isotope_names,
                              f"Gaussian σ={best_sigma}, n={n_inc}", combined_mask)
    save_fig(fig, f"NB2_02a_maps_n{n_inc}")

    fig = plot_difference_maps(abund, gt_abundances, isotope_names,
                               f"Gaussian σ={best_sigma}, n={n_inc}", combined_mask)
    save_fig(fig, f"NB2_02a_diff_n{n_inc}")

In [ ]:
# --- Exp 2a: SAMMY validation ---
from scipy.signal import savgol_filter
from pleiades.imaging.orchestrator import BatchFittingOrchestrator
from pleiades.imaging.models import PixelSpectrum

# Precompute denoised cubes (avoid re-denoising inside pixel loop)
gauss_dn_cubes = {}
for n_inc in noise_levels:
    gauss_dn_cubes[n_inc] = denoise_cube_gaussian(noisy_data[n_inc], best_sigma)

# Build SAMMY pixels for Gaussian experiment
sammy_pixels = []
sammy_meta = {}  # (row, col) -> metadata

# State offsets for row encoding:
# 0 = Clean, 1 = Noisy raw, 2 = Gauss raw, 3 = Gauss+T-dom recon
STATE_LABELS = ["Clean", "Noisy raw", "Gauss raw", "Gauss+T-dom recon"]

for pix_idx, (pix_name, (pr, pc)) in enumerate(REP_PIXELS.items()):
    for ni_idx, n_inc in enumerate(noise_levels):
        states = {}

        # State 0: Clean
        states[0] = np.clip(hyperspectral.data[:, pr, pc], -0.05, 1.05)

        # State 1: Noisy raw
        states[1] = np.clip(noisy_data[n_inc][:, pr, pc], -0.05, 1.05)

        # State 2: Gaussian denoised raw (precomputed)
        states[2] = np.clip(gauss_dn_cubes[n_inc][:, pr, pc], -0.05, 1.05)

        # State 3: Gaussian denoised -> T-domain recon
        coeffs_gauss = gauss_full[n_inc][0][:, pr, pc]
        if np.all(np.isfinite(coeffs_gauss)):
            T_recon = np.clip(np.exp(-A_matrix @ coeffs_gauss), 0.001, 1.0)
        else:
            T_recon = np.clip(gauss_dn_cubes[n_inc][:, pr, pc], 0.001, 1.0)
        states[3] = T_recon

        for state_idx, T_spec in states.items():
            row_enc = pix_idx * 100 + state_idx
            col_enc = ni_idx
            sigma_spec = np.full_like(T_spec, 0.01)
            px = PixelSpectrum(
                row=row_enc, col=col_enc,
                energy=energy, transmission=T_spec.astype(np.float64),
                uncertainty=sigma_spec.astype(np.float64),
                metadata={
                    "pixel_name": pix_name,
                    "state": STATE_LABELS[state_idx],
                    "n_incident": n_inc if state_idx > 0 else 0,
                    "pixel_rc": (pr, pc),
                },
            )
            sammy_pixels.append(px)
            sammy_meta[(row_enc, col_enc)] = px.metadata

print(f"SAMMY batch: {len(sammy_pixels)} pixel-spectra")

# Run SAMMY
orch = BatchFittingOrchestrator(
    imaging_config=config, sammy_executable=sammy_executable, n_workers=12)
results_2a = orch.fit_pixels(pixels=sammy_pixels, timeout_per_job=120.0, max_retries=1)
print(f"SAMMY returned {len(results_2a)} results")

# --- Build results table ---
import pandas as pd

# First extract clean reference
sammy_clean_ref = {}  # pix_name -> {"U-235": ppm, "Pu-241": ppm, "chi2": val}

rows_table = []
for res in results_2a:
    key = (res.row, res.col)
    if key not in sammy_meta:
        continue
    meta = sammy_meta[key]
    abund = res.get_abundances()
    total = sum(abund)
    u235_ppm = abund[0] / total * PPM if total > 0 else 0.0
    pu241_ppm = abund[1] / total * PPM if total > 0 else 0.0
    chi2 = res.chi_squared if res.chi_squared is not None else np.nan

    if meta["state"] == "Clean":
        sammy_clean_ref[meta["pixel_name"]] = {
            "U-235": u235_ppm, "Pu-241": pu241_ppm, "chi2": chi2}

    rows_table.append({
        "Pixel": meta["pixel_name"],
        "State": meta["state"],
        "n_inc": meta["n_incident"],
        "U-235 (ppm)": f"{u235_ppm:,.0f}",
        "Pu-241 (ppm)": f"{pu241_ppm:,.0f}",
        "chi2": f"{chi2:,.1f}" if np.isfinite(chi2) else "N/A",
    })

# Add deltas
for row in rows_table:
    ref = sammy_clean_ref.get(row["Pixel"], {})
    if ref and row["State"] != "Clean":
        u_ref = ref["U-235"]
        p_ref = ref["Pu-241"]
        u_val = float(row["U-235 (ppm)"].replace(",", ""))
        p_val = float(row["Pu-241 (ppm)"].replace(",", ""))
        row["δ U-235"] = f"{u_val - u_ref:+,.0f}"
        row["δ Pu-241"] = f"{p_val - p_ref:+,.0f}"
    else:
        row["δ U-235"] = "(ref)"
        row["δ Pu-241"] = "(ref)"

df_2a = pd.DataFrame(rows_table)
# Sort by pixel, n_inc, state order
state_order = {s: i for i, s in enumerate(STATE_LABELS)}
df_2a["_sort"] = df_2a["State"].map(state_order)
df_2a = df_2a.sort_values(["Pixel", "n_inc", "_sort"]).drop(columns=["_sort"])
print("\n=== Exp 2a: SAMMY Validation (Gaussian) ===")
print(df_2a.to_string(index=False))

### Exp 2a Observations

**Composition maps (T-domain fitting on Gaussian-smoothed cube):**
- Best sigma = 2.0 px, with a broad minimum — sigma 1-3 all cluster around 103-105K ppm MAE.
  Larger sigma (5.0) hurts: over-smoothing blurs spatial structure back into noise.
- At n=2: MAE 102,827 ppm (**+27.4%** improvement over raw 141,632 ppm).
- At n=10: MAE 70,599 ppm (+13.1% over raw 81,237 ppm) — less dramatic because
  raw noise is already lower, so there is less to gain from spatial averaging.

**SAMMY validation:**
- **Gauss raw** dramatically reduces delta vs noisy raw across all pixels. The Overlap pixel
  at n=10 is nearly perfect (delta = +1,786 ppm, chi2 ~24K — comparable to clean chi2 ~26K).
- Pure U-235 at n=10: Gauss raw delta = +22,184 ppm vs noisy raw +105,679 ppm — 5x reduction.
- **Gauss+T-dom recon hurts SAMMY** — same NB1 pattern. The parametric reconstruction
  drives chi2 down (112-3,146) but introduces large composition bias, especially at Pure U-235
  where the recon *inverts* the direction (delta flips from +67K to -391K at n=2).
  Low chi2 with wrong abundances = the model fits a smooth curve through the wrong physics.
- Gaussian denoising preserves spectral resonance shape (visible in spectra overlay) because
  it operates spatially within each energy bin. The residual vs clean is reduced relative to
  noisy raw but the resonance dip structure is intact.

## Experiment 2b: Non-Local Means Denoising

Non-local means (NLM) denoises by averaging similar patches across the image.
Unlike Gaussian smoothing, it preserves edges because dissimilar patches are
downweighted. The `h` parameter controls filtering strength (higher = more smoothing).

We sweep `h` = [0.05, 0.1, 0.2, 0.3, 0.5] with `fast_mode=True` and `patch_distance=4`.

In [ ]:
# --- Exp 2b: NLM h sweep at n=2 ---
nlm_hs = [0.05, 0.1, 0.2, 0.3, 0.5]
nlm_sweep = {}  # h -> {n_inc: (coeffs, cost, mask, abund, mae)}

n_inc_sweep = 2
print(f"NLM h sweep at n={n_inc_sweep}")
print(f"{'h':>6} {'MAE U-235 (ppm)':>18} {'MAE Pu-241 (ppm)':>18} {'Time (s)':>10}")
print("-" * 56)

best_h = None
best_nlm_mae = np.inf

for h_val in nlm_hs:
    t0 = time.time()
    cube_dn = denoise_cube_nlmeans(noisy_data[n_inc_sweep], h=h_val)
    t_denoise = time.time() - t0
    coeffs, cost, mask = fit_image_tdomain(
        cube_dn, sigma_data[n_inc_sweep], A_matrix,
        label=f"NLM h={h_val}")
    abund = coeffs_to_abundances(coeffs, mask)
    mae_u = compute_mae(abund[0], gt_abundances[0], mask & gt_mask) * PPM
    mae_p = compute_mae(abund[1], gt_abundances[1], mask & gt_mask) * PPM
    elapsed = time.time() - t0
    print(f"{h_val:>6.2f} {mae_u:>18,.0f} {mae_p:>18,.0f} {elapsed:>10.1f} "
          f"(denoise: {t_denoise:.1f}s)")
    nlm_sweep[h_val] = {n_inc_sweep: (coeffs, cost, mask, abund, mae_u)}

    if mae_u < best_nlm_mae:
        best_nlm_mae = mae_u
        best_h = h_val

print(f"\nBest h: {best_h} (MAE={best_nlm_mae:,.0f} ppm)")

# --- Sweep plot ---
fig, ax = plt.subplots(figsize=(8, 5))
maes_u = [nlm_sweep[h][n_inc_sweep][4] for h in nlm_hs]
ax.plot(nlm_hs, maes_u, "s-", color="#2ca02c", lw=2, markersize=8)
raw_mae_2 = compute_mae(raw_results[n_inc_sweep][3][0], gt_abundances[0],
                        raw_results[n_inc_sweep][2] & gt_mask) * PPM
ax.axhline(raw_mae_2, color="red", ls="--", alpha=0.7,
           label=f"Raw baseline ({raw_mae_2:,.0f} ppm)")
ax.set_xlabel("NLM h (filtering strength)")
ax.set_ylabel("MAE U-235 (ppm)")
ax.set_title(f"Exp 2b: NLM sweep at n={n_inc_sweep}")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
save_fig(fig, "NB2_02b_nlm_sweep")

# --- Spectral overlay at best h ---
cube_best_nlm = denoise_cube_nlmeans(noisy_data[n_inc_sweep], h=best_h)
fig = plot_spectra_at_pixels(
    energy,
    [hyperspectral.data, noisy_data[n_inc_sweep], cube_best_nlm],
    ["Clean", f"Noisy n={n_inc_sweep}", f"NLM h={best_h}"],
    title_suffix=f" — NLM h={best_h}, n={n_inc_sweep}")
save_fig(fig, "NB2_02b_nlm_spectra")

In [ ]:
# --- Exp 2b: Full-image maps at n=2 and n=10 with best h ---
nlm_full = {}  # n_inc -> (coeffs, cost, mask, abund)

for n_inc in noise_levels:
    if n_inc == n_inc_sweep and best_h in nlm_sweep:
        c, co, m, a, _ = nlm_sweep[best_h][n_inc]
        nlm_full[n_inc] = (c, co, m, a)
    else:
        print(f"Fitting NLM h={best_h} at n={n_inc}...")
        t0 = time.time()
        cube_dn = denoise_cube_nlmeans(noisy_data[n_inc], h=best_h)
        print(f"  Denoising took {time.time() - t0:.1f}s")
        coeffs, cost, mask = fit_image_tdomain(
            cube_dn, sigma_data[n_inc], A_matrix,
            label=f"NLM n={n_inc}")
        abund = coeffs_to_abundances(coeffs, mask)
        nlm_full[n_inc] = (coeffs, cost, mask, abund)

for n_inc in noise_levels:
    _, _, mask, abund = nlm_full[n_inc]
    combined_mask = mask & gt_mask

    mae_u = compute_mae(abund[0], gt_abundances[0], combined_mask) * PPM
    mae_p = compute_mae(abund[1], gt_abundances[1], combined_mask) * PPM
    raw_mae_u = compute_mae(raw_results[n_inc][3][0], gt_abundances[0],
                            raw_results[n_inc][2] & gt_mask) * PPM
    pct_improve = (raw_mae_u - mae_u) / raw_mae_u * 100
    print(f"n={n_inc}: NLM MAE U-235={mae_u:,.0f} ppm "
          f"(raw={raw_mae_u:,.0f}, improvement={pct_improve:+.1f}%)")

    fig = plot_abundance_maps(abund, isotope_names,
                              f"NLM h={best_h}, n={n_inc}", combined_mask)
    save_fig(fig, f"NB2_02b_maps_n{n_inc}")

    fig = plot_difference_maps(abund, gt_abundances, isotope_names,
                               f"NLM h={best_h}, n={n_inc}", combined_mask)
    save_fig(fig, f"NB2_02b_diff_n{n_inc}")

In [ ]:
# --- Exp 2b: SAMMY validation (cumulative with 2a) ---

# Precompute NLM denoised cubes
nlm_dn_cubes = {}
for n_inc in noise_levels:
    t0 = time.time()
    nlm_dn_cubes[n_inc] = denoise_cube_nlmeans(noisy_data[n_inc], h=best_h)
    print(f"  NLM denoise n={n_inc}: {time.time() - t0:.1f}s")

sammy_pixels_2b = []
sammy_meta_2b = {}

# States: 0=Clean, 1=Noisy raw, 2=NLM raw, 3=NLM+T-dom recon
STATE_LABELS_2B = ["Clean", "Noisy raw", "NLM raw", "NLM+T-dom recon"]

for pix_idx, (pix_name, (pr, pc)) in enumerate(REP_PIXELS.items()):
    for ni_idx, n_inc in enumerate(noise_levels):
        states = {}

        # State 0: Clean
        states[0] = np.clip(hyperspectral.data[:, pr, pc], -0.05, 1.05)

        # State 1: Noisy raw
        states[1] = np.clip(noisy_data[n_inc][:, pr, pc], -0.05, 1.05)

        # State 2: NLM denoised raw (precomputed)
        states[2] = np.clip(nlm_dn_cubes[n_inc][:, pr, pc], -0.05, 1.05)

        # State 3: NLM denoised -> T-domain recon
        coeffs_nlm = nlm_full[n_inc][0][:, pr, pc]
        if np.all(np.isfinite(coeffs_nlm)):
            T_recon = np.clip(np.exp(-A_matrix @ coeffs_nlm), 0.001, 1.0)
        else:
            T_recon = np.clip(nlm_dn_cubes[n_inc][:, pr, pc], 0.001, 1.0)
        states[3] = T_recon

        for state_idx, T_spec in states.items():
            row_enc = pix_idx * 100 + 10 + state_idx  # +10 offset to avoid 2a collision
            col_enc = ni_idx
            sigma_spec = np.full_like(T_spec, 0.01)
            px = PixelSpectrum(
                row=row_enc, col=col_enc,
                energy=energy, transmission=T_spec.astype(np.float64),
                uncertainty=sigma_spec.astype(np.float64),
                metadata={
                    "pixel_name": pix_name,
                    "state": STATE_LABELS_2B[state_idx],
                    "n_incident": n_inc if state_idx > 0 else 0,
                    "pixel_rc": (pr, pc),
                },
            )
            sammy_pixels_2b.append(px)
            sammy_meta_2b[(row_enc, col_enc)] = px.metadata

print(f"SAMMY batch 2b: {len(sammy_pixels_2b)} pixel-spectra")
orch_2b = BatchFittingOrchestrator(
    imaging_config=config, sammy_executable=sammy_executable, n_workers=12)
results_2b = orch_2b.fit_pixels(pixels=sammy_pixels_2b, timeout_per_job=120.0, max_retries=1)
print(f"SAMMY returned {len(results_2b)} results")

# Build table
rows_2b = []
for res in results_2b:
    key = (res.row, res.col)
    if key not in sammy_meta_2b:
        continue
    meta = sammy_meta_2b[key]
    abund = res.get_abundances()
    total = sum(abund)
    u235_ppm = abund[0] / total * PPM if total > 0 else 0.0
    pu241_ppm = abund[1] / total * PPM if total > 0 else 0.0
    chi2 = res.chi_squared if res.chi_squared is not None else np.nan

    rows_2b.append({
        "Pixel": meta["pixel_name"],
        "State": meta["state"],
        "n_inc": meta["n_incident"],
        "U-235 (ppm)": f"{u235_ppm:,.0f}",
        "Pu-241 (ppm)": f"{pu241_ppm:,.0f}",
        "chi2": f"{chi2:,.1f}" if np.isfinite(chi2) else "N/A",
    })

for row in rows_2b:
    ref = sammy_clean_ref.get(row["Pixel"], {})
    if ref and row["State"] != "Clean":
        u_ref = ref["U-235"]
        p_ref = ref["Pu-241"]
        u_val = float(row["U-235 (ppm)"].replace(",", ""))
        p_val = float(row["Pu-241 (ppm)"].replace(",", ""))
        row["δ U-235"] = f"{u_val - u_ref:+,.0f}"
        row["δ Pu-241"] = f"{p_val - p_ref:+,.0f}"
    else:
        row["δ U-235"] = "(ref)"
        row["δ Pu-241"] = "(ref)"

df_2b = pd.DataFrame(rows_2b)
state_order_2b = {s: i for i, s in enumerate(STATE_LABELS_2B)}
df_2b["_sort"] = df_2b["State"].map(state_order_2b)
df_2b = df_2b.sort_values(["Pixel", "n_inc", "_sort"]).drop(columns=["_sort"])
print("\n=== Exp 2b: SAMMY Validation (NLM) ===")
print(df_2b.to_string(index=False))

### Exp 2b Observations

**Composition maps:**
- NLM is remarkably insensitive to the `h` parameter: MAE ranges only 100,439-100,857 ppm
  across h=0.05-0.5. The adaptive patch-matching self-regulates smoothing strength.
- Best h = 0.3 at n=2: MAE 100,439 ppm (**+29.1%** over raw) — best single method so far.
- At n=10: MAE 59,026 ppm (**+27.3%** over raw) — the gap widens vs Gaussian (+13.1%).
  NLM's edge-preserving property pays off more when the underlying signal has sharper
  spatial structure to exploit.
- Runtime: ~16s denoise + ~89s fitting per sweep point. The denoise step is 15x slower than
  Gaussian (<1s) but still tractable for the 256x256 image.

**SAMMY validation:**
- **NLM raw** mean |delta| = 125,317 ppm — better than Gauss raw (151,254 ppm) for SAMMY too.
- At Pure Pu-241 n=10: NLM raw gets remarkably close — delta +85,704 ppm with the correct
  direction (Pu-241 dominant at 938K ppm). Compare noisy raw which has a *negative* U-235
  composition (-693K ppm) — completely unphysical.
- Pure U-235 n=10: NLM raw delta = +19,103 ppm — excellent, nearly matching clean reference.
- **NLM+T-dom recon**: same pattern as 2a — T-domain reconstruction inflates delta to 260K
  mean despite lower chi2. The parametric model overfits to a smooth approximation that
  discards the resonance fine structure SAMMY needs.
- The spectral overlay confirms NLM preserves resonance dips while reducing the noise envelope.
  Residuals are smaller than noisy raw but retain the same frequency content (no spectral
  smoothing artifacts).

## Experiment 2c: Wavelet BayesShrink Denoising

Wavelet denoising uses multi-resolution analysis to separate signal from noise.
BayesShrink adaptively estimates thresholds per wavelet subband.

We compare wavelet families: `db1` (Haar), `db2`, and `sym4`.

In [ ]:
# --- Exp 2c: Wavelet family sweep at n=2 ---
wavelet_families = ["db1", "db2", "sym4"]
wavelet_sweep = {}  # wavelet -> {n_inc: (coeffs, cost, mask, abund, mae)}

n_inc_sweep = 2
print(f"Wavelet sweep at n={n_inc_sweep}")
print(f"{'wavelet':>8} {'MAE U-235 (ppm)':>18} {'MAE Pu-241 (ppm)':>18} {'Time (s)':>10}")
print("-" * 58)

best_wavelet = None
best_wav_mae = np.inf

for wname in wavelet_families:
    t0 = time.time()
    cube_dn = denoise_cube_wavelet(noisy_data[n_inc_sweep], wavelet=wname)
    t_denoise = time.time() - t0
    coeffs, cost, mask = fit_image_tdomain(
        cube_dn, sigma_data[n_inc_sweep], A_matrix,
        label=f"Wav {wname}")
    abund = coeffs_to_abundances(coeffs, mask)
    mae_u = compute_mae(abund[0], gt_abundances[0], mask & gt_mask) * PPM
    mae_p = compute_mae(abund[1], gt_abundances[1], mask & gt_mask) * PPM
    elapsed = time.time() - t0
    print(f"{wname:>8} {mae_u:>18,.0f} {mae_p:>18,.0f} {elapsed:>10.1f} "
          f"(denoise: {t_denoise:.1f}s)")
    wavelet_sweep[wname] = {n_inc_sweep: (coeffs, cost, mask, abund, mae_u)}

    if mae_u < best_wav_mae:
        best_wav_mae = mae_u
        best_wavelet = wname

print(f"\nBest wavelet: {best_wavelet} (MAE={best_wav_mae:,.0f} ppm)")

# --- Sweep plot ---
fig, ax = plt.subplots(figsize=(8, 5))
maes_u = [wavelet_sweep[w][n_inc_sweep][4] for w in wavelet_families]
x_pos = range(len(wavelet_families))
ax.bar(x_pos, maes_u, color="#ff7f0e", alpha=0.8)
ax.set_xticks(x_pos)
ax.set_xticklabels(wavelet_families)
raw_mae_2 = compute_mae(raw_results[n_inc_sweep][3][0], gt_abundances[0],
                        raw_results[n_inc_sweep][2] & gt_mask) * PPM
ax.axhline(raw_mae_2, color="red", ls="--", alpha=0.7,
           label=f"Raw baseline ({raw_mae_2:,.0f} ppm)")
ax.set_xlabel("Wavelet family")
ax.set_ylabel("MAE U-235 (ppm)")
ax.set_title(f"Exp 2c: Wavelet sweep at n={n_inc_sweep}")
ax.legend()
ax.grid(True, alpha=0.3, axis="y")
fig.tight_layout()
save_fig(fig, "NB2_02c_wavelet_sweep")

# --- Spectral overlay at best wavelet ---
cube_best_wav = denoise_cube_wavelet(noisy_data[n_inc_sweep], wavelet=best_wavelet)
fig = plot_spectra_at_pixels(
    energy,
    [hyperspectral.data, noisy_data[n_inc_sweep], cube_best_wav],
    ["Clean", f"Noisy n={n_inc_sweep}", f"Wavelet {best_wavelet}"],
    title_suffix=f" — Wavelet {best_wavelet}, n={n_inc_sweep}")
save_fig(fig, "NB2_02c_wavelet_spectra")

In [ ]:
# --- Exp 2c: Full-image maps at n=2 and n=10 with best wavelet ---
wavelet_full = {}  # n_inc -> (coeffs, cost, mask, abund)

for n_inc in noise_levels:
    if n_inc == n_inc_sweep and best_wavelet in wavelet_sweep:
        c, co, m, a, _ = wavelet_sweep[best_wavelet][n_inc]
        wavelet_full[n_inc] = (c, co, m, a)
    else:
        print(f"Fitting wavelet {best_wavelet} at n={n_inc}...")
        t0 = time.time()
        cube_dn = denoise_cube_wavelet(noisy_data[n_inc], wavelet=best_wavelet)
        print(f"  Denoising took {time.time() - t0:.1f}s")
        coeffs, cost, mask = fit_image_tdomain(
            cube_dn, sigma_data[n_inc], A_matrix,
            label=f"Wav n={n_inc}")
        abund = coeffs_to_abundances(coeffs, mask)
        wavelet_full[n_inc] = (coeffs, cost, mask, abund)

for n_inc in noise_levels:
    _, _, mask, abund = wavelet_full[n_inc]
    combined_mask = mask & gt_mask

    mae_u = compute_mae(abund[0], gt_abundances[0], combined_mask) * PPM
    mae_p = compute_mae(abund[1], gt_abundances[1], combined_mask) * PPM
    raw_mae_u = compute_mae(raw_results[n_inc][3][0], gt_abundances[0],
                            raw_results[n_inc][2] & gt_mask) * PPM
    pct_improve = (raw_mae_u - mae_u) / raw_mae_u * 100
    print(f"n={n_inc}: Wavelet MAE U-235={mae_u:,.0f} ppm "
          f"(raw={raw_mae_u:,.0f}, improvement={pct_improve:+.1f}%)")

    fig = plot_abundance_maps(abund, isotope_names,
                              f"Wavelet {best_wavelet}, n={n_inc}", combined_mask)
    save_fig(fig, f"NB2_02c_maps_n{n_inc}")

    fig = plot_difference_maps(abund, gt_abundances, isotope_names,
                               f"Wavelet {best_wavelet}, n={n_inc}", combined_mask)
    save_fig(fig, f"NB2_02c_diff_n{n_inc}")

In [ ]:
# --- Exp 2c: SAMMY validation (Wavelet) ---

# Precompute wavelet denoised cubes
wav_dn_cubes = {}
for n_inc in noise_levels:
    t0 = time.time()
    wav_dn_cubes[n_inc] = denoise_cube_wavelet(noisy_data[n_inc], wavelet=best_wavelet)
    print(f"  Wavelet denoise n={n_inc}: {time.time() - t0:.1f}s")

sammy_pixels_2c = []
sammy_meta_2c = {}

STATE_LABELS_2C = ["Clean", "Noisy raw", "Wavelet raw", "Wavelet+T-dom recon"]

for pix_idx, (pix_name, (pr, pc)) in enumerate(REP_PIXELS.items()):
    for ni_idx, n_inc in enumerate(noise_levels):
        states = {}

        # State 0: Clean
        states[0] = np.clip(hyperspectral.data[:, pr, pc], -0.05, 1.05)

        # State 1: Noisy raw
        states[1] = np.clip(noisy_data[n_inc][:, pr, pc], -0.05, 1.05)

        # State 2: Wavelet denoised raw (precomputed)
        states[2] = np.clip(wav_dn_cubes[n_inc][:, pr, pc], -0.05, 1.05)

        # State 3: Wavelet denoised -> T-domain recon
        coeffs_wav = wavelet_full[n_inc][0][:, pr, pc]
        if np.all(np.isfinite(coeffs_wav)):
            T_recon = np.clip(np.exp(-A_matrix @ coeffs_wav), 0.001, 1.0)
        else:
            T_recon = np.clip(wav_dn_cubes[n_inc][:, pr, pc], 0.001, 1.0)
        states[3] = T_recon

        for state_idx, T_spec in states.items():
            row_enc = pix_idx * 100 + 20 + state_idx  # +20 offset for 2c
            col_enc = ni_idx
            sigma_spec = np.full_like(T_spec, 0.01)
            px = PixelSpectrum(
                row=row_enc, col=col_enc,
                energy=energy, transmission=T_spec.astype(np.float64),
                uncertainty=sigma_spec.astype(np.float64),
                metadata={
                    "pixel_name": pix_name,
                    "state": STATE_LABELS_2C[state_idx],
                    "n_incident": n_inc if state_idx > 0 else 0,
                    "pixel_rc": (pr, pc),
                },
            )
            sammy_pixels_2c.append(px)
            sammy_meta_2c[(row_enc, col_enc)] = px.metadata

print(f"SAMMY batch 2c: {len(sammy_pixels_2c)} pixel-spectra")
orch_2c = BatchFittingOrchestrator(
    imaging_config=config, sammy_executable=sammy_executable, n_workers=12)
results_2c = orch_2c.fit_pixels(pixels=sammy_pixels_2c, timeout_per_job=120.0, max_retries=1)
print(f"SAMMY returned {len(results_2c)} results")

# Build table
rows_2c = []
for res in results_2c:
    key = (res.row, res.col)
    if key not in sammy_meta_2c:
        continue
    meta = sammy_meta_2c[key]
    abund = res.get_abundances()
    total = sum(abund)
    u235_ppm = abund[0] / total * PPM if total > 0 else 0.0
    pu241_ppm = abund[1] / total * PPM if total > 0 else 0.0
    chi2 = res.chi_squared if res.chi_squared is not None else np.nan

    rows_2c.append({
        "Pixel": meta["pixel_name"],
        "State": meta["state"],
        "n_inc": meta["n_incident"],
        "U-235 (ppm)": f"{u235_ppm:,.0f}",
        "Pu-241 (ppm)": f"{pu241_ppm:,.0f}",
        "chi2": f"{chi2:,.1f}" if np.isfinite(chi2) else "N/A",
    })

for row in rows_2c:
    ref = sammy_clean_ref.get(row["Pixel"], {})
    if ref and row["State"] != "Clean":
        u_ref = ref["U-235"]
        p_ref = ref["Pu-241"]
        u_val = float(row["U-235 (ppm)"].replace(",", ""))
        p_val = float(row["Pu-241 (ppm)"].replace(",", ""))
        row["δ U-235"] = f"{u_val - u_ref:+,.0f}"
        row["δ Pu-241"] = f"{p_val - p_ref:+,.0f}"
    else:
        row["δ U-235"] = "(ref)"
        row["δ Pu-241"] = "(ref)"

df_2c = pd.DataFrame(rows_2c)
state_order_2c = {s: i for i, s in enumerate(STATE_LABELS_2C)}
df_2c["_sort"] = df_2c["State"].map(state_order_2c)
df_2c = df_2c.sort_values(["Pixel", "n_inc", "_sort"]).drop(columns=["_sort"])
print("\n=== Exp 2c: SAMMY Validation (Wavelet) ===")
print(df_2c.to_string(index=False))

### Exp 2c Observations

**Composition maps:**
- Wavelet family matters: db1 (Haar) at 112,784 ppm is notably worse than db2 (103,079) and
  sym4 (102,465). The Haar wavelet's blocky basis functions leave artifacts; smoother wavelets
  (db2, sym4) better match the image's spatial structure.
- Best wavelet = sym4 at n=2: MAE 102,465 ppm (+27.7% over raw) — essentially tied with
  Gaussian sigma=2 (102,827 ppm).
- At n=10: MAE 66,011 ppm (+18.7%) — between Gaussian (70,599) and NLM (59,026).
- Wavelet denoising is extremely fast (~1s for full cube) — 15x faster than NLM.

**SAMMY validation:**
- **Wavelet raw** achieves the best mean |delta| of all methods: **116,109 ppm** — beating
  NLM raw (125,317) and Gauss raw (151,254).
- The standout result is Pure U-235 at n=2: Wavelet raw delta = **-2,824 ppm** — essentially
  zero error. The wavelet preserved the resonance structure almost perfectly at this pixel.
- Pure Pu-241 remains the hardest pixel for all methods — but Wavelet raw at n=10 gets
  delta = +286,602 ppm vs noisy raw -669,164 ppm (correct sign, 2.3x reduction).
- **Wavelet+T-dom recon**: again inflates delta (mean 272K) — confirming the universal
  pattern: T-domain parametric reconstruction is the bottleneck for SAMMY, regardless
  of what denoising precedes it.
- Spectral overlay shows wavelet denoised spectra hug the clean curve more tightly than
  Gaussian — the adaptive thresholding removes noise while preserving sharp resonance dips
  without the ring artifacts that Gaussian introduces near steep transitions.

## Grand Comparison

Side-by-side comparison of all methods: Ground Truth, Raw T-domain, Gaussian, NLM, Wavelet.
Summary metrics across all methods and noise levels.

In [ ]:
# --- Grand visual comparison: GT / Raw / Gauss / NLM / Wavelet ---

for n_inc in noise_levels:
    method_data = {
        "Ground Truth": (gt_abundances, gt_mask),
        "Raw T-domain": (raw_results[n_inc][3], raw_results[n_inc][2]),
        f"Gauss σ={best_sigma}": (gauss_full[n_inc][3], gauss_full[n_inc][2]),
        f"NLM h={best_h}": (nlm_full[n_inc][3], nlm_full[n_inc][2]),
        f"Wav {best_wavelet}": (wavelet_full[n_inc][3], wavelet_full[n_inc][2]),
    }

    n_methods = len(method_data)
    n_iso = len(isotope_names)
    fig, axes = plt.subplots(
        n_iso, n_methods, figsize=(4 * n_methods, 4.5 * n_iso),
        gridspec_kw={"wspace": 0.05, "hspace": 0.15},
    )

    for col_idx, (mname, (abund, mask)) in enumerate(method_data.items()):
        combined = mask & gt_mask
        for iso_idx, iso_name in enumerate(isotope_names):
            ax = axes[iso_idx, col_idx]
            amap = abund[iso_idx].copy() * PPM
            amap[~combined] = np.nan
            im = ax.imshow(amap, cmap="viridis", origin="upper",
                          vmin=0, vmax=PPM)
            if iso_idx == 0:
                ax.set_title(mname, fontsize=10)
            if col_idx == 0:
                ax.set_ylabel(iso_name, fontsize=11)
            ax.set_xticks([])
            ax.set_yticks([])

    # Single colorbar on the right, not overlapping images
    fig.subplots_adjust(right=0.92)
    cbar_ax = fig.add_axes([0.93, 0.15, 0.015, 0.7])
    fig.colorbar(im, cax=cbar_ax, label="Composition (ppm)")

    fig.suptitle(f"Grand Comparison — n={n_inc}", y=0.98, fontsize=14)
    save_fig(fig, f"NB2_03_grand_comparison_n{n_inc}")

In [ ]:
# --- Summary metrics table ---
print("=" * 80)
print("SUMMARY: All methods × noise levels")
print("=" * 80)

summary_rows = []
for n_inc in noise_levels:
    method_data = {
        "Raw T-domain": (raw_results[n_inc][3], raw_results[n_inc][2]),
        f"Gauss σ={best_sigma}": (gauss_full[n_inc][3], gauss_full[n_inc][2]),
        f"NLM h={best_h}": (nlm_full[n_inc][3], nlm_full[n_inc][2]),
        f"Wav {best_wavelet}": (wavelet_full[n_inc][3], wavelet_full[n_inc][2]),
    }

    for mname, (abund, mask) in method_data.items():
        combined = mask & gt_mask
        mae_u = compute_mae(abund[0], gt_abundances[0], combined) * PPM
        mae_p = compute_mae(abund[1], gt_abundances[1], combined) * PPM
        rmse_u = compute_rmse(abund[0], gt_abundances[0], combined) * PPM
        rmse_p = compute_rmse(abund[1], gt_abundances[1], combined) * PPM
        summary_rows.append({
            "n_inc": n_inc,
            "Method": mname,
            "MAE U-235": f"{mae_u:,.0f}",
            "MAE Pu-241": f"{mae_p:,.0f}",
            "RMSE U-235": f"{rmse_u:,.0f}",
            "RMSE Pu-241": f"{rmse_p:,.0f}",
            "_mae_u": mae_u,
        })

df_summary = pd.DataFrame(summary_rows)
print(df_summary.drop(columns=["_mae_u"]).to_string(index=False))

# --- Bar chart ---
fig, axes = plt.subplots(1, len(noise_levels), figsize=(8 * len(noise_levels), 6))
if len(noise_levels) == 1:
    axes = [axes]

colors_bar = ["#d62728", "#1f77b4", "#2ca02c", "#ff7f0e"]
method_names_short = ["Raw", f"Gauss", f"NLM", f"Wavelet"]

for ax_idx, n_inc in enumerate(noise_levels):
    ax = axes[ax_idx]
    subset = df_summary[df_summary["n_inc"] == n_inc]
    maes = subset["_mae_u"].values
    x = range(len(maes))
    bars = ax.bar(x, maes, color=colors_bar[:len(maes)], alpha=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(method_names_short[:len(maes)], rotation=15)
    ax.set_ylabel("MAE U-235 (ppm)")
    ax.set_title(f"n={n_inc}")
    ax.grid(True, alpha=0.3, axis="y")
    # Add value labels
    for bar, val in zip(bars, maes):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
                f"{val:,.0f}", ha="center", va="bottom", fontsize=9)

fig.suptitle("NB2: Spatial Denoising — MAE Comparison", y=1.02)
fig.tight_layout()
save_fig(fig, "NB2_04_summary_metrics")

In [ ]:
# --- SAMMY Grand Summary: all experiments combined ---
print("=" * 100)
print("SAMMY GRAND SUMMARY — All spatial denoising methods")
print("=" * 100)

# Combine all SAMMY tables
all_sammy_rows = []

# Exp 2a results
for _, row in df_2a.iterrows():
    all_sammy_rows.append({
        "Exp": "2a-Gauss",
        "Pixel": row["Pixel"],
        "State": row["State"],
        "n_inc": row["n_inc"],
        "U-235 (ppm)": row["U-235 (ppm)"],
        "Pu-241 (ppm)": row["Pu-241 (ppm)"],
        "δ U-235": row["δ U-235"],
        "δ Pu-241": row["δ Pu-241"],
        "chi2": row["chi2"],
    })

# Exp 2b results (exclude Clean/Noisy raw duplicates)
for _, row in df_2b.iterrows():
    if row["State"] in ("Clean", "Noisy raw"):
        continue
    all_sammy_rows.append({
        "Exp": "2b-NLM",
        "Pixel": row["Pixel"],
        "State": row["State"],
        "n_inc": row["n_inc"],
        "U-235 (ppm)": row["U-235 (ppm)"],
        "Pu-241 (ppm)": row["Pu-241 (ppm)"],
        "δ U-235": row["δ U-235"],
        "δ Pu-241": row["δ Pu-241"],
        "chi2": row["chi2"],
    })

# Exp 2c results (exclude Clean/Noisy raw duplicates)
for _, row in df_2c.iterrows():
    if row["State"] in ("Clean", "Noisy raw"):
        continue
    all_sammy_rows.append({
        "Exp": "2c-Wavelet",
        "Pixel": row["Pixel"],
        "State": row["State"],
        "n_inc": row["n_inc"],
        "U-235 (ppm)": row["U-235 (ppm)"],
        "Pu-241 (ppm)": row["Pu-241 (ppm)"],
        "δ U-235": row["δ U-235"],
        "δ Pu-241": row["δ Pu-241"],
        "chi2": row["chi2"],
    })

df_grand_sammy = pd.DataFrame(all_sammy_rows)
print(df_grand_sammy.to_string(index=False))

# --- Quick comparison: mean |delta| per method-state ---
print("\n" + "=" * 60)
print("Mean |δ U-235| by Method-State (lower = closer to clean SAMMY)")
print("=" * 60)
for exp_label in ["2a-Gauss", "2b-NLM", "2c-Wavelet"]:
    sub = df_grand_sammy[df_grand_sammy["Exp"] == exp_label]
    for state in sub["State"].unique():
        ss = sub[sub["State"] == state]
        deltas = []
        for _, r in ss.iterrows():
            d = r["δ U-235"]
            if d != "(ref)":
                deltas.append(abs(float(d.replace(",", ""))))
        if deltas:
            print(f"  {exp_label:>12} | {state:<25} | mean |δ|={np.mean(deltas):,.0f} ppm")

## Final Observations

### NB2 Hypothesis: Confirmed

Spatially denoising the raw transmission cube before fitting **works** — all three methods
reduce composition map MAE by 27-29% at n=2 and 13-27% at n=10, and they all preserve
spectral resonance structure for SAMMY (unlike NB1's T-domain reconstruction which destroyed it).

### Composition Map Rankings

| Method | n=2 MAE (ppm) | n=10 MAE (ppm) | Speed |
|--------|---------------|-----------------|-------|
| Raw T-domain | 141,632 | 81,237 | baseline |
| Gaussian σ=2 | 102,827 (+27%) | 70,599 (+13%) | <1s denoise |
| Wavelet sym4 | 102,465 (+28%) | 66,011 (+19%) | ~1s denoise |
| **NLM h=0.3** | **100,439 (+29%)** | **59,026 (+27%)** | ~16s denoise |

NLM wins composition maps at both noise levels, with a decisive advantage at n=10 where
its edge-preserving property allows 27% improvement vs Gaussian's 13%.

### SAMMY Rankings (mean |delta U-235| in ppm, lower = better)

| State | Gauss | NLM | Wavelet |
|-------|-------|-----|---------|
| Noisy raw (baseline) | 368,116 | 368,116 | 368,116 |
| Denoised raw | 151,254 | 125,317 | **116,109** |
| Denoised + T-dom recon | 276,041 | 260,513 | 272,722 |

**Wavelet wins SAMMY** with the lowest mean delta, while NLM wins composition maps.
The T-domain reconstruction universally hurts SAMMY (~270K mean delta regardless of prior
denoising) — confirming the NB1 finding that parametric reconstruction erases the resonance
fine structure SAMMY needs for isotope discrimination.

### Key Insight: Two Separate Tasks, Two Different Winners

- **Composition maps** (NNLS/T-domain fitting): NLM is best. Its patch-based averaging
  produces spatially smoother maps with better edge preservation.
- **SAMMY fitting** (full physics): Wavelet is best. BayesShrink removes noise while
  preserving sharp spectral features more faithfully than spatial averaging methods.
- These are not contradictory — NNLS/T-domain fitting benefits from smooth input (less
  noise variance → less fitting error), while SAMMY benefits from *accurate* input
  (preserved resonance shapes → correct isotope discrimination).

### The T-domain Reconstruction Trap (Universal)

Across all three experiments, the denoised+T-dom-recon state produces deceptively low chi2
values (7-3,787) but *worse* composition accuracy than feeding SAMMY the denoised raw data
directly (chi2 13K-55K). Low chi2 with wrong physics is worse than high chi2 with preserved
spectral structure. **Never feed SAMMY parametric reconstructions.**

### Recommendations for NB3

1. **Carry forward NLM h=0.3** as the spatial preprocessing step for composition maps.
2. **Carry forward Wavelet sym4** as the preprocessing step for SAMMY fitting.
3. Consider a **hybrid pipeline**: Wavelet denoise → SAMMY for per-pixel physics,
   NLM denoise → T-domain for composition maps. The denoising is cheap relative to fitting.
4. Explore whether combining spatial denoising with NB1's TV post-processing on composition
   maps provides additive benefit (spatial preprocessing + spatial postprocessing).